# 7. Explore annotated MSI datasets in METASPACE

This notebook searches public METASPACE metadata without downloading imzML/ibd files. Start broadly to learn the values present in the database, inspect biological and acquisition metadata, open individual dataset pages, and only then refine the filters. The exported JSON is consumed by the existing `query --filters` command.

In [ ]:
from pathlib import Path

from IPython.display import display

from msi_autoencoder_wrapper.dataset_management.exploration import DatasetExplorer

explorer = DatasetExplorer(source="metaspace")

## Inspect supported filters

The source reports both provider-side GraphQL filters and local quantitative filters. Dataset identity and biological/acquisition fields are filtered by METASPACE. Annotation counts, unique-molecule counts, optical-image presence, and `exclude_dataset_ids` are evaluated after the provider query.

In [ ]:
explorer.get_available_filters()

## Start with a broad query

METASPACE contains many public mouse datasets. A broad first query avoids assuming spelling or metadata values that are not actually present. The result table includes all review fields, including condition, pixel count, image dimensions, annotation count at the configured FDR, annotation databases, submitter/project information, and a direct dataset URL.

In [ ]:
broad_filters = {
    "organism": "Mouse",
    "exclude_dataset_ids": [],
}
results = explorer.filter(broad_filters)
display(results.head(20))
print(f"Found {len(results)} datasets")

Summarize values present in the returned records before choosing stricter filters. Empty values mean that the submitter did not provide that metadata.

In [ ]:
for column in ["organism_parts", "condition", "polarity", "processing_status", "databases"]:
    print(f"\n{column}")
    display(results[column].value_counts(dropna=False).head(20))

## Refine the provider query

Use values observed above. This example applies native dataset filters equivalent to the METASPACE web UI, then requires at least one annotation at 10% FDR. `include_molecule_stats` downloads annotation rows only (no ion images) and calculates deduplicated and selection-unique formula/adduct identities.

In [ ]:
filters = {
    "organism": "Mouse",
    "name": "liver",
    "organism_part": "Liver",
    "polarity": "Negative",
    "condition": "Wild type",
    "annotation_fdr": 0.1,
    "min_annotation_count": 1,
    "include_molecule_stats": True,
    "exclude_dataset_ids": [],
}
results = explorer.filter(filters)
display(results)

`rejected()` contains datasets removed by local quantitative constraints, with the observed value and a direct review URL. Datasets eliminated by native METASPACE filters never leave the provider and therefore do not appear here.

In [ ]:
explorer.rejected()

## Inspect one complete source record

The table is intentionally compact. Retrieve a selected record to inspect all metadata supplied by METASPACE before accepting it.

In [ ]:
if not results.empty:
    dataset_id = results.iloc[0]["dataset_id"]
    dataset_record = explorer.source.get_dataset_metadata(dataset_id)
    display(dataset_record)
    print(results.iloc[0]["project_url"])

## Exclude reviewed datasets and export

Open the URLs and list unsuitable IDs below. Manual exclusions are stored in the exported configuration and applied by both the notebook explorer and the CLI query.

In [ ]:
excluded_dataset_ids = []
if excluded_dataset_ids:
    explorer.exclude(excluded_dataset_ids)
display(explorer.results())

In [ ]:
output_path = Path("assets/configs/datasets/metaspace_mouse_liver.json")
explorer.export_config(output_path)

## Use the exported configuration

Run `manage_datasets.py query --source metaspace --filters assets/configs/datasets/metaspace_mouse_liver.json --selection workspace/datasets/selections/metaspace_mouse_liver.json`. Review the selection, then pass it to `download --source metaspace`. Querying and metadata review do not download MSI binary data.